# Prompt LLMs (Inference)

This notebook implements **Zero-Shot Inference**, the first of three approaches tested in my project.

Instead of training or modifying a model, I use GPT-4o-mini directly via the OpenAI API. I pass each tax question from my dataset as a prompt and collect the model's answers. This serves as my **baseline**: how well can a general-purpose LLM answer Austrian tax law questions with no additional training or retrieval?

The output is a CSV file (`inference_results.csv`) containing the model's answer for each question ID, ready to be evaluated with BERTScore.

In [ ]:
import pandas as pd # Used to load and manipulate the CSV dataset as a table (DataFrame)
from openai import OpenAI # The official OpenAI Python client for making API calls
import time # Used to add a small delay between requests to stay within API rate limits
import csv # Used for the QUOTE_ALL constant to safely format CSV output

# Connect to the OpenAI API using my secret key.
# All subsequent API calls will be authenticated through this client object.
client = OpenAI(api_key = "api_key")

### Phase 1: Data Loading

Here I load my cleaned question dataset into memory. The CSV contains 644 Austrian tax law questions, each with a unique `id` and a `prompt` field holding the question text.

I also initialize an empty `results` list. As I loop through the questions, I append each `{id, answer}` pair to this list, which I then convert to a DataFrame and save to CSV at the end.

In [7]:
# Load the cleaned dataset from disk into a pandas DataFrame.
# Each row represents one tax question with columns: 'id', 'prompt', 'correct_answer', etc.
df = pd.read_csv("dataset_clean.csv")

# Initialize an empty list that will collect my results as I loop.
# Each entry will be a dict: {"id": ..., "answer": ...}
results = []

### Phase 2: The Processing Loop

This section iterates through every row of my CSV. It includes two types of **guardrails**.

The first is an **empty cell guard**. If a question cell is `NaN` (blank), I skip it rather than sending an empty prompt to the API, which would waste API credits and produce a meaningless answer.

The second is an **exception handler**. If the API call fails for any reason (network timeout, rate limit hit), I catch the error, log it, and write a placeholder `"ERROR"` string so the row still exists in my output. This means the loop never crashes midway through 644 rows.

After the loop, I convert the collected results into a DataFrame and save it as a CSV using `QUOTE_ALL`. This wrapping of every cell in double quotes is critical: Austrian legal answers frequently contain commas (e.g. `§ 1, Abs. 2`), which would otherwise be misread as column separators and corrupt the 644-row structure.

In [ ]:
# Iterate over every row in the DataFrame.
# 'index' is the row number (0-643), 'row' is a Series with all column values for that row.
for index, row in df.iterrows():

    # Extract the unique question identifier (used later to merge with ground truth during evaluation)
    question_id = row['id']

    # Extract the tax law question text that will be sent to the model
    question_text = row['prompt']

    # Guardrail 1: Skip rows where the question is missing (NaN) to avoid sending blank prompts to the API
    if pd.isna(question_text):
        continue

    try:
        # Send the question to the OpenAI Chat Completions API
        response = client.chat.completions.create(
            model = "gpt-4o-mini", # Lightweight and cost-efficient GPT-4 variant
            messages = [
                # System message: defines the model's role, behavior, and constraints.
                # It instructs the model to act as an Austrian tax law expert,
                # always cite the relevant legal paragraph, verify the correct law applies
                # (e.g. UStG for VAT, EStG for income tax), and answer only in German
                # using accurate Austrian legal terminology.
                {"role": "system", "content": "Du bist ein Experte für das österreichische Steuerrecht. Analysiere bei jeder Frage die spezifische Steuerkategorie (z. B. EStG, KStG, UStG, etc.) und gib eine definitive Antwort basierend auf österreichischen Rechtsstandards. Verifiziere vor der Antwort, dass der zitierte Paragraf tatsächlich zum Thema passt (z. B. dass Umsatzsteuerfragen das UStG zitieren). Nenne immer die entsprechenden Gesetzesstellen. Falls sich Gesetze kürzlich geändert haben, priorisiere die aktuellsten Regelungen für 2025/2026. Antworte ausschließlich auf Deutsch und verwende die korrekte österreichische Rechtsterminologie."},

                # User message: the actual tax question from this row of my dataset
                {"role": "user", "content": question_text}
            ]
        )

        # Extract the model's text answer from the API response object.
        # The response is structured as: response.choices[0].message.content
        answer = response.choices[0].message.content

        # Append the question ID and the generated answer as a dict to my results list
        results.append({"id": question_id, "answer": answer})

        # Print progress to the console so I can track how far through the 644 rows I am
        print(f"Processed {index+1}/644: {question_id}")

        # Rate limiting: wait 0.1 seconds between each request to stay within OpenAI's API limits
        time.sleep(0.1)

    except Exception as e:
        # Guardrail 2: If any error occurs (network issue, API failure),
        # log it to the console and write a placeholder so this row is not lost entirely.
        # This prevents the entire loop from crashing on a single bad request.
        print(f"Error at ID {question_id}: {e}")
        results.append({"id": question_id, "answer": "ERROR: Could not generate answer."})

# Convert the list of {id, answer} dicts into a pandas DataFrame for easy export
output_df = pd.DataFrame(results)

# Save the results to a CSV file.
# quoting=1 (csv.QUOTE_ALL) wraps every cell in double quotes.
# This is essential for legal text, which often contains commas (e.g. '§ 1, Abs. 2').
# Without this, those commas would be misinterpreted as column separators,
# corrupting the CSV structure and breaking the 644-row count.
output_df.to_csv("inference_results.csv", index = False, sep = ",", quoting = 1) # quoting=1 is the same as csv.QUOTE_ALL

# Confirm successful completion
print("Done! File saved as inference_results.csv")

Processed 1/644: CORP-TAX-001
Processed 2/644: CORP-TAX-002
